In [7]:
from abc import ABC, abstractmethod
import numpy as np
import pandas as pd

# Get mean for each input

In [135]:
class BaseModel(ABC):
    @abstractmethod
    def train(self, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass

class MeanModel(BaseModel):

    def train(self, y):
        self.mean = np.mean(y)

    def predict(self, X):
        return [self.mean]*len(X)

# Descriptive Statistics


**Takeaway**:
- Base can have multiple layers, think about how to design it
- Base defines instance that can be called using self. in all child functions
- Within child functions, any other child function can be called by function().regardless of hierarchy
- Add error check at the beginning of the function

In [ ]:
X = np.array([
    [2.0],
    [1.0],
    [3.0],
    [4.0],
    [3.0]
])

# Target vector y (4 samples)
y = np.array([5.0, 7.0, 9.0, 11.0, 6.0])

In [183]:
class Statistic(ABC):
    pass

class UnivariateStatistics(Statistic):
    @abstractmethod
    def compute(self, data):
        pass


class Mean(UnivariateStatistics):
    def compute(self, data):
        return np.mean(data)
    
class Median(UnivariateStatistics):
    def compute(self, data):
        data = np.sort(data)
        n = len(data)
        if n%2 == 0:
            return (data[n//2] + data[n//2-1])/2
        else:
            return data[n//2] 

class Mode(UnivariateStatistics):
    def compute(self, data):
        flat = data.flatten()
        count = {}
        for i in flat:
            count[i] = count.get(i, 0) + 1
        return max(count, key = count.get)

# Var(X) = (1/n) * Σ (xᵢ - μ)² for population
# Var(X) = (1/n-1) * Σ (xᵢ - μ)² for sample
# np.var(X) for population
# np.var(X, ddof = 1) for sample
class Variance(UnivariateStatistics):
    def compute(self, data):
        return np.sum(np.square(data - np.mean(data)))/len(data)

#std = √variance
#np.std()
class STD(UnivariateStatistics):
    def compute(self, data):
        return np.sqrt(Variance().compute(data))

#A q-quantile is the value where q fraction of the data lies below it.
#position = (n-1)*q
class Quantile(UnivariateStatistics):
    def __init__(self, q):
        self.q = q
    def compute(self, data):
        return np.quantile(data, self.q)


class MultivariateStatistics(Statistic):
    @abstractmethod
    def compute(self, X, y):
        pass

# Cov(X,Y) = (1/n) Σ (xi − μx)(yi − μy)
# tells us direction of relationship, but not scale
# -inf to inf
class Covariance(MultivariateStatistics):
    def compute(self, x, y):
        if len(x) != len(y):
            raise ValueError('X and y must have the same length')
        x_flat = x.flatten()
        x_mean = np.mean(x_flat)
        y_mean = np.mean(y)

        return  np.sum((x_flat-x_mean)*(y-y_mean))/len(x)

# ρ = Cov(X,Y) / (σx σy)
# [-1, 1]
class Correlation(MultivariateStatistics):
    def compute(self, x, y):
        if len(x) != len(y):
            raise ValueError('X and y must have the same length')
        cov = Covariance().compute(x, y)
        std_x = STD().compute(x)
        std_y = STD().compute(y)

        return cov / (std_x * std_y)

In [184]:
test = Mean()
test.compute(X)

np.float64(2.6)

In [185]:
test = Median()
test.compute(X)

array([3.])

In [186]:
test = Mode()
test.compute(X)

np.float64(3.0)

In [187]:
test = Variance()
test.compute(X)

np.float64(1.0400000000000003)

In [188]:
test = STD()
test.compute(X)

np.float64(1.019803902718557)

In [189]:
test = Quantile(0.5)
test.compute(X)

np.float64(3.0)

In [190]:
test = Covariance()
test.compute(X,y)

np.float64(1.44)

In [191]:
test = Correlation()
test.compute(X,y)

np.float64(0.6555213366563067)

# Hypothesis Testing

## ABC Start

class HypothesisTest(ABC):

    @abstractmethod
    def statistic(self, sample1, sample2):
        pass

    @abstractmethod
    def p_value(self):
        pass


## Z Test
Formula:
z = (x̄ − μ) / (σ / √n)


## Two Sample T Test
Formula:
t = (x̄1 − x̄2) / √(s1²/n1 + s2²/n2)


## Chi Square Test
Formula:
χ² = Σ ((Observed − Expected)² / Expected)


## Permutation Test
Formula:
shuffle labels → recompute statistic → compare distribution

# A/B Testing / Experimentation

## ABC Start

class Experiment(ABC):

    @abstractmethod
    def fit(self, control, treatment):
        pass

    @abstractmethod
    def effect(self):
        pass


## Difference in Means
Formula:
Δ = mean_treatment − mean_control


## Standard Error
Formula:
SE = √(s1²/n1 + s2²/n2)


## Confidence Interval
Formula:
x̄ ± z * (σ / √n)


## Bootstrap Confidence Interval
Formula:
resample with replacement → compute statistic distribution → percentile bounds

# Probability Distributions

## ABC Start

class Distribution(ABC):

    @abstractmethod
    def pdf(self, x):
        pass

    @abstractmethod
    def sample(self, n):
        pass


## Bernoulli Distribution
Formula:
P(X=1) = p  
P(X=0) = 1 − p


## Binomial Distribution
Formula:
P(X=k) = (n choose k) p^k (1−p)^(n−k)


## Poisson Distribution
Formula:
P(X=k) = (λ^k e^(−λ)) / k!


## Normal Distribution
Formula:
f(x) = (1/(σ√2π)) * exp(-(x−μ)²/(2σ²))


## Exponential Distribution
Formula:
f(x) = λ e^(−λx)

# Machine Learning Models

## ABC Start

class Model(ABC):

    @abstractmethod
    def fit(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass


## Linear Regression
Formula:
β = (XᵀX)⁻¹Xᵀy


## Mean Squared Error
Formula:
MSE = (1/n) * Σ (yi − ŷi)²


## Logistic Regression
Formula:
σ(x) = 1 / (1 + e^(−x))


## Logistic Loss
Formula:
−Σ [ y log(p) + (1−y) log(1−p) ]


## K Means Clustering
Formula:
minimize Σ ||xi − μk||²

# Evaluation Metrics

## ABC Start

class Metric(ABC):

    @abstractmethod
    def compute(self, y_true, y_pred):
        pass


## Accuracy
Formula:
Accuracy = (TP + TN) / Total


## Precision
Formula:
Precision = TP / (TP + FP)


## Recall
Formula:
Recall = TP / (TP + FN)


## F1 Score
Formula:
F1 = 2 * (Precision * Recall) / (Precision + Recall)


## ROC AUC
Formula:
Area under ROC curve